<a href="https://colab.research.google.com/github/RashiBista/GraphRAG/blob/main/03_extract_2wiki.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip -q install groq

In [3]:
import json, re, time
from pathlib import Path
from google.colab import drive, userdata
from groq import Groq, RateLimitError, BadRequestError

In [4]:
drive.mount("/content/drive")
WORKDIR = Path("/content/drive/MyDrive/HybridGraphRAG/twowiki")

def load_jsonl(path):
    with path.open("r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

chunks = load_jsonl(WORKDIR / "document_chunks.jsonl")
questions = load_jsonl(WORKDIR / "questions.jsonl")

groq_client = Groq(api_key=userdata.get("GROQ_API_KEY"))
LLM = "openai/gpt-oss-20b"   # the real extraction model, not the safeguard one

def chat_with_retry(**kwargs):
    for attempt in range(8):
        try:
            return groq_client.chat.completions.create(**kwargs)
        except RateLimitError:
            wait = 3 * (attempt + 1)
            print(f"  rate limited, waiting {wait}s...")
            time.sleep(wait)
    raise RuntimeError("Still rate limited after 8 retries")

print(len(chunks), "chunks,", len(questions), "questions")

Mounted at /content/drive
182 chunks, 20 questions


In [5]:
ENTITY_TYPES = {
    "PERSON":       "a real person",
    "WORK":         "a film, song, book, album or other creative work",
    "PLACE":        "a city, region, country or other location",
    "ORGANIZATION": "a company, school, band, club or institution",
    "AWARD":        "a prize or honour",
    "DATE":         "a date or year, copied as written in the text",
    "OTHER":        "important, but fits none of the above",
}

# Direction rule: (source, RELATION, target). The definitions say who is who.
RELATION_TYPES = {
    # links between things (these are what connect articles to each other)
    "DIRECTOR":               "target directed source (source is a film)",
    "FATHER":                 "target is the father of source",
    "MOTHER":                 "target is the mother of source",
    "SPOUSE":                 "target is married to source",
    "CHILD":                  "target is a child of source",
    "SIBLING":                "target is a brother or sister of source",
    "PERFORMER":              "target performed source (e.g. the singer of a song)",
    "COMPOSER":               "target composed source",
    "EDUCATED_AT":            "source studied at target",
    "EMPLOYER":               "source works or worked for target",
    "AWARD_RECEIVED":         "source received target (an award)",
    "PLACE_OF_BIRTH":         "source was born in target",
    "PLACE_OF_DEATH":         "source died in target",
    "PLACE_OF_BURIAL":        "source was buried in target",
    "COUNTRY_OF_CITIZENSHIP": "source is a citizen of target",
    "COUNTRY_OF_ORIGIN":      "source (a work) comes from target (a country)",
    "COUNTRY":                "source is located in target (a country)",
    # facts whose target is a DATE
    "DATE_OF_BIRTH":          "source was born on target (a DATE)",
    "DATE_OF_DEATH":          "source died on target (a DATE)",
    "PUBLICATION_DATE":       "source was released or published on target (a DATE)",
    "INCEPTION":              "source was founded or created on target (a DATE)",
    # catch-all
    "RELATED_TO":             "a clear link that fits none of the above",
}
print(len(ENTITY_TYPES), "entity types,", len(RELATION_TYPES), "relation types")

7 entity types, 22 relation types


In [6]:
def extract_with_retry(text, max_attempts=3):
    for attempt in range(1, max_attempts + 1):
        try:
            result = extract(text)
            return normalize_extraction(result)

        except BadRequestError as e:
            print(
                f"Structured-output failure "
                f"(attempt {attempt}/{max_attempts}): {e}"
            )

            if attempt < max_attempts:
                time.sleep(2 * attempt)

        except Exception as e:
            print(f"Unexpected extraction error: {e}")
            raise

    raise RuntimeError(
        "Extraction failed after all retry attempts."
    )

In [7]:
import json, re

def build_schema(entity_types, relation_types):
    return {
        "type": "object",
        "properties": {
            "entities": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "name": {"type": "string"},
                        "type": {"type": "string", "enum": list(entity_types)},
                    },
                    "required": ["name", "type"],
                    "additionalProperties": False,
                },
            },
            "relationships": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "source": {"type": "string"},
                        "relation": {"type": "string", "enum": list(relation_types)},
                        "target": {"type": "string"},
                    },
                    "required": ["source", "relation", "target"],
                    "additionalProperties": False,
                },
            },
        },
        "required": ["entities", "relationships"],
        "additionalProperties": False,
    }

SCHEMA = build_schema(ENTITY_TYPES, RELATION_TYPES)

def build_prompt(text):
    types_txt = "\n".join(f"- {k}: {v}" for k, v in ENTITY_TYPES.items())
    rels_txt = "\n".join(f"- {k}: {v}" for k, v in RELATION_TYPES.items())
    return f"""You are an information extraction system. Build a knowledge graph from the encyclopedia-style TEXT below.

The TEXT begins with the title of its article ("Title: ..."). The article is about the entity named in the title, so "he", "she", "it", "the film" or "the band" refer to that entity.

ENTITY TYPES:
{types_txt}

RELATIONS, written as (source, RELATION, target). Use only these:
{rels_txt}

Rules:
1. Extract only what the TEXT explicitly states. Never use outside knowledge, even if you know more about the subject.
2. Entity names are noun phrases taken from the text or the title, using the fullest name given. Never use verbs or actions.
3. Follow the relation definitions exactly. They say which way each relation points.
4. Dates are DATE entities, copied as written.
5. Every source and target of a relationship must also appear in the entities list.
6. Prefer a specific relation over RELATED_TO. If nothing fits, return empty lists.

TEXT:
{text}
"""

def extract(text, model=None):
    response = chat_with_retry(
        model=model or LLM,
        messages=[
            {
                "role": "user",
                "content": build_prompt(text)
            }
        ],
        temperature=0,
        reasoning_effort="low",
        max_completion_tokens=4096,
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "graph_extraction",
                "strict": True,
                "schema": SCHEMA
            }
        }
    )

    content = response.choices[0].message.content

    if not content:
        raise ValueError("The model returned empty content.")

    return json.loads(content)

def validate_extraction(data):
    if not isinstance(data, dict):
        return False
    if not isinstance(data.get("entities"), list) or not isinstance(data.get("relationships"), list):
        return False
    for e in data["entities"]:
        if not isinstance(e, dict) or "name" not in e or "type" not in e:
            return False
    for r in data["relationships"]:
        if not isinstance(r, dict) or not {"source", "relation", "target"} <= r.keys():
            return False
    return True

def normalize_entity_name(name):
    return re.sub(r"\s+", " ", str(name)).strip()

def normalize_extraction(data):
    entities, relationships = [], []
    seen_e, seen_r = set(), set()
    for e in data.get("entities", []):
        name = normalize_entity_name(e["name"])
        etype = str(e["type"]).strip().upper()
        if name and (name.lower(), etype) not in seen_e:
            entities.append({"name": name, "type": etype})
            seen_e.add((name.lower(), etype))
    for r in data.get("relationships", []):
        s = normalize_entity_name(r["source"])
        rel = str(r["relation"]).strip().upper()
        t = normalize_entity_name(r["target"])
        if s and t and rel and (s.lower(), rel, t.lower()) not in seen_r:
            relationships.append({"source": s, "relation": rel, "target": t})
            seen_r.add((s.lower(), rel, t.lower()))
    return {"entities": entities, "relationships": relationships}

In [8]:
TEST_TITLES = ["Naïma Ben Ali", "Zine El Abidine Ben Ali", "Gyrid of Sweden", "Harald Bluetooth"]
by_title = {c["title"]: c for c in chunks}

def show_extraction(ch):
    print(f"\n===== {ch['title']} [{ch['id']}] =====")

    try:
        out = extract_with_retry(ch["text"])

        print("Valid shape:", validate_extraction(out))

        print("ENTITIES:")
        for e in out["entities"]:
            print(f"  {e['name']} [{e['type']}]")

        print("RELATIONSHIPS:")
        for r in out["relationships"]:
            print(
                f"  {r['source']} "
                f"--{r['relation']}--> {r['target']}"
            )

        return out

    except RuntimeError as e:
        print("FAILED:", e)
        return None

In [9]:
show_extraction(by_title[TEST_TITLES[0]])


===== Naïma Ben Ali [0101_na_ma_ben_ali] =====
Valid shape: True
ENTITIES:
  Naïma Ben Ali [PERSON]
  Zine El Abidine Ben Ali [PERSON]
  Tunisia [PLACE]
  1987 [DATE]
  1988 [DATE]
RELATIONSHIPS:
  Naïma Ben Ali --SPOUSE--> Zine El Abidine Ben Ali


{'entities': [{'name': 'Naïma Ben Ali', 'type': 'PERSON'},
  {'name': 'Zine El Abidine Ben Ali', 'type': 'PERSON'},
  {'name': 'Tunisia', 'type': 'PLACE'},
  {'name': '1987', 'type': 'DATE'},
  {'name': '1988', 'type': 'DATE'}],
 'relationships': [{'source': 'Naïma Ben Ali',
   'relation': 'SPOUSE',
   'target': 'Zine El Abidine Ben Ali'}]}

In [10]:
results = {}

for title in TEST_TITLES:
    print(f"\n{'='*60}")
    result = show_extraction(by_title[title])
    results[title] = result



===== Naïma Ben Ali [0101_na_ma_ben_ali] =====
Valid shape: True
ENTITIES:
  Naïma Ben Ali [PERSON]
  Zine El Abidine Ben Ali [PERSON]
  Tunisia [PLACE]
  1987 [DATE]
  1988 [DATE]
RELATIONSHIPS:
  Naïma Ben Ali --SPOUSE--> Zine El Abidine Ben Ali


===== Zine El Abidine Ben Ali [0103_zine_el_abidine_ben_ali] =====
Valid shape: True
ENTITIES:
  Zine El Abidine Ben Ali [PERSON]
  Habib Bourguiba [PERSON]
  Leïla Ben Ali [PERSON]
  Saudi Arabia [PLACE]
  Jeddah [PLACE]
  Tunisia [PLACE]
  Interpol [ORGANIZATION]
  Tunisian court [ORGANIZATION]
  3 September 1936 [DATE]
  19 September 2019 [DATE]
  1987 [DATE]
  October 1987 [DATE]
  7 November 1987 [DATE]
  25 October 2009 [DATE]
  14 January 2011 [DATE]
  20 June 2011 [DATE]
  June 2012 [DATE]
  April 2013 [DATE]
RELATIONSHIPS:
  Zine El Abidine Ben Ali --DATE_OF_BIRTH--> 3 September 1936
  Zine El Abidine Ben Ali --DATE_OF_DEATH--> 19 September 2019
  Zine El Abidine Ben Ali --PLACE_OF_DEATH--> Jeddah
  Zine El Abidine Ben Ali --COUN

In [11]:
FAILED_TITLES = [
    "Zine El Abidine Ben Ali",
    "Harald Bluetooth"
]

for titles in FAILED_TITLES:
  ch = by_title[title]

  print("TITLE:", title)
  print("CHUNK ID:", ch["id"])
  print("CHARACTERS:", len(ch["text"]))
  print("\nACTUAL TEXT:\n")
  print(ch["text"])

TITLE: Harald Bluetooth
CHUNK ID: 0141_harald_bluetooth
CHARACTERS: 589

ACTUAL TEXT:

Harald Bluetooth: Harald "Bluetooth" Gormsson (died c. 985/86) was a king of Denmark and Norway. He was the son of King Gorm the Old and of Thyra Dannebod. Harald ruled as king of Denmark from c. 958 – c. 986. Harald introduced Christianity to Denmark and consolidated his rule over most of Jutland and Zealand. Harald's rule as king of Norway following the assassination of King Harald Greycloak of Norway was more tenuous, most likely lasting for no more than a few years in the 970s. Some sources say his son Sweyn Forkbeard forcibly deposed him from his Danish throne before his death.
TITLE: Harald Bluetooth
CHUNK ID: 0141_harald_bluetooth
CHARACTERS: 589

ACTUAL TEXT:

Harald Bluetooth: Harald "Bluetooth" Gormsson (died c. 985/86) was a king of Denmark and Norway. He was the son of King Gorm the Old and of Thyra Dannebod. Harald ruled as king of Denmark from c. 958 – c. 986. Harald introduced Christia

In [12]:
def extract_json_mode(text):
    response = chat_with_retry(
        model=LLM,
        messages=[
            {
                "role": "user",
                "content": build_prompt(text)
                + "\n\nReturn only valid JSON with exactly "
                  "two keys: entities and relationships."
            }
        ],
        temperature=0,
        response_format={"type": "json_object"}
    )

    raw = response.choices[0].message.content

    print("RAW MODEL RESPONSE:\n")
    print(raw)

    data = json.loads(raw)

    print("\nSTRUCTURAL VALIDATION:",
          validate_extraction(data))

    return normalize_extraction(data)

In [13]:
harald = by_title["Harald Bluetooth"]

test_response = groq_client.chat.completions.create(
    model=LLM,
    messages=[
        {
            "role": "user",
            "content": build_prompt(harald["text"])
        }
    ],
    temperature=0,
    max_completion_tokens=1500
)

message = test_response.choices[0].message

print("FINISH REASON:", test_response.choices[0].finish_reason)
print("CONTENT:", repr(message.content))
print("REASONING:", repr(message.reasoning))
print("USAGE:", test_response.usage)

FINISH REASON: length
CONTENT: ''
REASONING: 'We need to build knowledge graph. Entities: Harald Bluetooth, King Gorm the Old, Thyra Dannebod, Denmark, Norway, Jutland, Zealand, King Harald Greycloak of Norway, Sweyn Forkbeard. Also maybe "Christianity" but it\'s a concept, not a person or work. But we can include as OTHER? But relation? "introduced Christianity to Denmark" could be a relation? Not in list. So maybe ignore.\n\nWe need to list entities with types. Determine types: Harald Bluetooth is PERSON. King Gorm the Old is PERSON. Thyra Dannebod is PERSON. Denmark is PLACE. Norway is PLACE. Jutland is PLACE. Zealand is PLACE. King Harald Greycloak of Norway is PERSON. Sweyn Forkbeard is PERSON. Also "Christianity" is OTHER but no relation.\n\nAlso "c. 985/86" is DATE. "c. 958 – c. 986" is DATE? Actually a range. We can treat as DATE? The relation PUBLICATION_DATE? Not relevant. We can use DATE_OF_BIRTH? No. We can use DATE_OF_DEATH for Harald: died c. 985/86. Also maybe DATE_OF_BI

In [14]:
def extract_without_json_mode(text):
    response = chat_with_retry(
        model=LLM,
        messages=[
            {
                "role": "user",
                "content": build_prompt(text)
                + "\n\nReturn only valid JSON. Do not include "
                  "Markdown or explanations."
            }
        ],
        temperature=0
    )

    raw = response.choices[0].message.content

    print("RAW MODEL RESPONSE:\n")
    print(raw)

    return raw

In [15]:
ch = by_title["Harald Bluetooth"]
r = groq_client.chat.completions.create(
    model=LLM,
    messages=[{"role": "user", "content": build_prompt(ch["text"])}],
    temperature=0,
    max_completion_tokens=4000,
)
c = r.choices[0]
print("FINISH:", c.finish_reason)
print("CONTENT:", repr(c.message.content)[:300])
print("REASONING len:", len(c.message.reasoning or ""))
print("USAGE:", r.usage)

FINISH: length
CONTENT: ''
REASONING len: 15878
USAGE: CompletionUsage(completion_tokens=4000, prompt_tokens=792, total_tokens=4792, completion_time=4.485261572, completion_tokens_details=CompletionTokensDetails(reasoning_tokens=3998), prompt_time=0.056825214, prompt_tokens_details=None, queue_time=0.88171994, total_time=4.5420867860000005)


In [24]:
from pydantic import BaseModel
from typing import Literal

EType = Literal[tuple(ENTITY_TYPES)]
RType = Literal[tuple(RELATION_TYPES)]

class Ent(BaseModel):
    name: str
    type: EType

class Rel(BaseModel):
    source: str
    relation: RType
    target: str

class Extraction(BaseModel):
    entities: list[Ent]
    relationships: list[Rel]

OUT   = WORKDIR / "graph_records.jsonl"
USAGE = WORKDIR / "extraction_usage.jsonl"

# (model, reasoning_effort, max_completion_tokens), tried in order
LADDER = [("openai/gpt-oss-20b", "low", 3000),
          ("openai/gpt-oss-20b", "medium", 6000),
          ("openai/gpt-oss-120b", "low", 3000)]

def extract_once(text, model, effort, max_tok):
    r = chat_with_retry(
        model=model, temperature=0,
        reasoning_effort=effort,          # if Groq rejects this param, drop it
        max_completion_tokens=max_tok,
        messages=[{"role": "user", "content": build_prompt(text)}],
        response_format={"type": "json_schema",
                         "json_schema": {"name": "graph_extraction",
                                         "strict": True, "schema": SCHEMA}},
    )
    content = r.choices[0].message.content
    if not content:
        raise ValueError(f"empty content, finish={r.choices[0].finish_reason}")
    return Extraction.model_validate_json(content), r.usage

def run_extraction(chunks):
    done = set()
    if OUT.exists():
        done = {json.loads(l)["chunk_id"] for l in OUT.open(encoding="utf-8")}
    with OUT.open("a", encoding="utf-8") as fo, USAGE.open("a", encoding="utf-8") as fu:
        for i, ch in enumerate(chunks):
            if ch["id"] in done:
                continue
            ext, used_model, err = None, None, None
            for model, effort, max_tok in LADDER:
                try:
                    ext, usage = extract_once(ch["text"], model, effort, max_tok)
                    used_model = model
                    fu.write(json.dumps({"chunk_id": ch["id"], "model": model,
                             "prompt": usage.prompt_tokens,
                             "completion": usage.completion_tokens,
                              "reasoning": usage.completion_tokens_details.reasoning_tokens}) + "\n")
                    break
                except Exception as e:
                    err = str(e)[:200]
            rec = {"chunk_id": ch["id"], "source": ch["title"], "model": used_model,
                   "error": None if ext else err,
                   **(normalize_extraction(ext.model_dump()) if ext
                      else {"entities": [], "relationships": []})}
            fo.write(json.dumps(rec, ensure_ascii=False) + "\n"); fo.flush(); fu.flush()
            print(f"{i+1}/{len(chunks)} {ch['title'][:40]:40} "
                  f"{'OK ' if ext else 'FAIL'} {used_model or err}")

In [25]:
test_response = groq_client.chat.completions.create(
    model=LLM,
    messages=[
        {
            "role": "user",
            "content": build_prompt(harald["text"])
        }
    ],
    temperature=0,
    max_completion_tokens=1500
)

message = test_response.choices[0].message

print("FINISH REASON:", test_response.choices[0].finish_reason)
print("CONTENT:", repr(message.content))
print("REASONING:", repr(message.reasoning))
print("USAGE:", test_response.usage)

FINISH REASON: length
CONTENT: ''
REASONING: 'We need to build knowledge graph. Entities: Harald Bluetooth, King Gorm the Old, Thyra Dannebod, Denmark, Norway, Jutland, Zealand, King Harald Greycloak of Norway, Sweyn Forkbeard. Also maybe "Christianity" but it\'s a concept not in types. But we can include as OTHER? But relation: "introduced Christianity to Denmark" - relation? None defined. Could use RELATED_TO. But rule: prefer specific relation. No relation for "introduced". So use RELATED_TO: (Harald Bluetooth, RELATED_TO, Christianity). But Christianity is not defined type. Could be OTHER. But we need entity list. So include entity "Christianity" as OTHER.\n\nAlso "c. 985/86" is a date. "c. 958 – c. 986" is a date range. We need separate DATE entities: "c. 985/86", "c. 958", "c. 986". Also "970s" maybe date? It\'s a decade. Could be DATE "970s". Also "c. 958 – c. 986" maybe two dates. We\'ll create separate DATE entities.\n\nRelations:\n\n- FATHER: (Harald Bluetooth, FATHER, King G

In [26]:
def probe(title, model, effort, max_tok=3000):
    ch = by_title[title]
    r = groq_client.chat.completions.create(
        model=model, temperature=0,
        reasoning_effort=effort, max_completion_tokens=max_tok,
        messages=[{"role": "user", "content": build_prompt(ch["text"])}],
        response_format={"type": "json_schema",
                         "json_schema": {"name": "graph_extraction",
                                         "strict": True, "schema": SCHEMA}},
    )
    c = r.choices[0]
    rt = r.usage.completion_tokens_details.reasoning_tokens
    print(f"{title[:28]:28} {model.split('/')[-1]:12} {effort:6} "
          f"finish={c.finish_reason:6} completion={r.usage.completion_tokens:5} "
          f"reasoning={rt:5} content_len={len(c.message.content or '')}")

for t in ["Harald Bluetooth", "Zine El Abidine Ben Ali"]:
    for m, e in [("openai/gpt-oss-20b", "low"),
                 ("openai/gpt-oss-120b", "low")]:
        probe(t, m, e)

Harald Bluetooth             gpt-oss-20b  low    finish=stop   completion= 1314 reasoning=  896 content_len=1578
Harald Bluetooth             gpt-oss-120b low    finish=stop   completion=  790 reasoning=  309 content_len=1467
Zine El Abidine Ben Ali      gpt-oss-20b  low    finish=stop   completion= 1195 reasoning=    6 content_len=4388
Zine El Abidine Ben Ali      gpt-oss-120b low    finish=stop   completion= 1160 reasoning=  632 content_len=1569
